In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

pg = pd.read_csv('data/pre_gold.csv').set_index('year_quarter')
pg = pg.drop(columns=['Unnamed: 0'], errors='ignore')


def monetary_rule(y, q, na):   return 0 if na else 1
def gdp_api_rule(y, q, na):   return 2
def cpi_rule(y, q, na):       return 0 if na else 1
def fiscal_api_rule(y, q, na): return 0 if na else (2 if y == 2010 else (4 if y >= 2023 else 1))
def labour_rule(y, q, na):    return 0 if na else 2
def annual_rule(y, q, na):    return 0 if na else 3
def annual_ext_rule(y, q, na): return 0 if na else (5 if y >= 2023 else 3)
def elec_rule(y, q, na):      return 0 if na else (3 if y < 2015 else 1)
def scrape_rule(y, q, na):    return 0 if na else 4
def rate_api_rule(y, q, na):  return 0 if na else 1

variables = [
    ('M0',                         'M0',                              'Monetary\n(raw)',       monetary_rule),
    ('M1',                         'M1',                              'Monetary\n(raw)',       monetary_rule),
    ('M2',                         'M2',                              'Monetary\n(raw)',       monetary_rule),
    ('Nominal GDP',                'gdp_2',                           'GDP\n(raw)',            gdp_api_rule),
    ('Real GDP (base 2021)',           'gdp_s21',                     'GDP\n(raw)',            gdp_api_rule),
    ('Population',                 'Population',                      'GDP\n(raw)',            annual_rule),
    ('Tax revenue',                'tax_revenue',                     'Fiscal\n(raw)',         fiscal_api_rule),
    ('Indirect tax',               'indirect_tax',                    'Fiscal\n(raw)',         fiscal_api_rule),
    ('Total expenditure',          'total_expense',                   'Fiscal\n(raw)',         fiscal_api_rule),
    ('Employed (thousands)',        'employed_people_thousands',       'Labour\n(raw)',         labour_rule),
    ('Labour force (thousands)',    'labour_force_people_thousands',   'Labour\n(raw)',         labour_rule),
    ('Unemployed (thousands)',      'unemployed_people_thousands',     'Labour\n(raw)',         labour_rule),
    ('Informal employed',          'informal_employed_15_70_thousands','Labour\n(raw)',         labour_rule),
    ('Rule of law',                'rule_of_law',                     'Institutional\n(raw)',  annual_ext_rule),
    ('Gov. effectiveness',         'government_effectiveness',        'Institutional\n(raw)',  annual_ext_rule),
    ('Corruption perceptions',     'corruption_perceptions',          'Institutional\n(raw)',  annual_ext_rule),
    ('Business freedom',           'business_freedom',                'Institutional\n(raw)',  annual_rule),
    ('Economic freedom',           'economic_freedom',                'Institutional\n(raw)',  annual_rule),
    ('Fiscal freedom',             'fiscal_freedom',                  'Institutional\n(raw)',  annual_rule),
    ('Schooling',                  'schooling',                       'Institutional\n(raw)',  annual_ext_rule),
    ('Electricity consumption',    'electricity_consumptions',        'Electricity',           elec_rule),
    ('Interest rate',              'interest_rate_avg_q',             'Rates',                 scrape_rule),
    ('USD exchange rate',          'usd_avg_q',                       'Rates',                 rate_api_rule),
    ('EUR exchange rate',          'eur_avg_q',                       'Rates',                 rate_api_rule),
    ('Cash demand = M0/M2',        'cash_demand',                     'Derived\nmetrics',      monetary_rule),
    ('Tax burden = TaxRev/GDP',    'tax_burden',                      'Derived\nmetrics',      fiscal_api_rule),
    ('Direct tax share',           'direct_tax_share',                'Derived\nmetrics',      fiscal_api_rule),
    ('Gov. exp. share = Exp/GDP',  'gov_expenditure_share',           'Derived\nmetrics',      fiscal_api_rule),
    ('GDP per capita = GDP/Pop',   'gdp_per_capita',                  'Derived\nmetrics',      annual_rule),
    ('Deflated M1 = M1/CPI',      'deflated_M1',                      'Derived\nmetrics',      monetary_rule),
    ('Deflated tax rev = TaxRev/CPI', 'deflated_tax_revenue',         'Derived\nmetrics',      fiscal_api_rule),
    ('Employment rate',            'employment_rate',                 'Derived\nmetrics',      labour_rule),
    ('LFP rate',                   'lfp_rate',                        'Derived\nmetrics',      labour_rule),
    ('Unemployment rate',          'unemployment_rate',               'Derived\nmetrics',      labour_rule),
]

quarters = pg.index.tolist()
n_q = len(quarters)
n_v = len(variables)

status = np.zeros((n_v, n_q), dtype=int)
labels = []
blocks = []

for i, (display, col, block, rule) in enumerate(variables):
    labels.append(display)
    blocks.append(block)
    for j, q in enumerate(quarters):
        year = int(q[:4])
        qn = int(q[-1])
        val = pg.loc[q, col] if col in pg.columns else np.nan
        is_na = pd.isna(val)
        status[i, j] = rule(year, qn, is_na)

colors = ['#d9534f', '#5cb85c', '#f0ad4e', '#9EAFC2', '#5bc0de', '#e8a0bf']
cmap = mcolors.ListedColormap(colors)
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm = mcolors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(17, 12))
ax.imshow(status, aspect='auto', cmap=cmap, norm=norm,
          interpolation='nearest')

xtick_idx = list(range(0, n_q, 4))
ax.set_xticks(xtick_idx)
ax.set_xticklabels([quarters[i] for i in xtick_idx],
                   rotation=45, ha='right', fontsize=8)

ax.set_yticks(range(n_v))
ax.set_yticklabels(labels, fontsize=8)

prev_block = blocks[0]
block_ranges = {prev_block: [0, 0]}
for i in range(1, n_v):
    if blocks[i] != prev_block:
        ax.axhline(i - 0.5, color='white', lw=2)
        prev_block = blocks[i]
        block_ranges[prev_block] = [i, i]
    else:
        block_ranges[prev_block][1] = i

for b, (start, end) in block_ranges.items():
    mid = (start + end) / 2
    ax.text(n_q + 0.8, mid, b.replace('\n', ' '),
            va='center', ha='left', fontsize=7.5,
            fontstyle='italic', color='#444')

derived_start = next(i for i, (_, _, b, _) in enumerate(variables)
                     if 'Derived' in b)
ax.axhline(derived_start - 0.5, color='black', lw=2.5, ls='-')
ax.text(-5, derived_start - 0.7, 'Derived →',
        fontsize=8, fontweight='bold', va='bottom', ha='right',
        color='#333')

q2015 = quarters.index('2015Q1')
q2022 = quarters.index('2022Q1')
ax.axvline(q2015 - 0.5, color='black', lw=0.8, ls='--', alpha=0.5)
ax.axvline(q2022 - 0.5, color='black', lw=0.8, ls='--', alpha=0.5)
ax.text(q2015 - 1, -2.3, 'APIs begin\n~2015',
        ha='center', fontsize=7, color='#333')
ax.text(q2022 - 1, -2.3, 'Full-scale\ninvasion 2022',
        ha='center', fontsize=7, color='#333')

legend_elements = [
    mpatches.Patch(facecolor='#5cb85c',
                   label='Official API (programmatic)'),
    mpatches.Patch(facecolor='#f0ad4e',
                   label='Historical Excel / manual'),
    mpatches.Patch(facecolor='#9EAFC2',
                   label='Annual data (repeated to quarters)'),
    mpatches.Patch(facecolor='#5bc0de',
                   label='Web scraping (MinFin)'),
    mpatches.Patch(facecolor='#e8a0bf',
                   label='Extrapolated (2023–24, carried forward)'),
    mpatches.Patch(facecolor='#d9534f',
                   label='Missing (not published)'),
]
ax.legend(handles=legend_elements, loc='upper center',
          bbox_to_anchor=(0.5, -0.08), ncol=3, fontsize=8.5,
          frameon=True)

ax.set_title('Data availability and ingestion method: '
             'raw inputs (top) and derived metrics (bottom)',
             fontsize=12, pad=14)
ax.set_xlim(-0.5, n_q - 0.5)
ax.set_ylim(n_v - 0.5, -0.5)

plt.tight_layout()
plt.subplots_adjust(bottom=0.14)
plt.savefig('outputs/data_availability_heatmap.png', dpi=150,
            bbox_inches='tight')
plt.close()